# Merged RAG Assistant

Combines two things that were previously in separate notebooks/scripts:

1. **From the LangChain/ESI script**: multi-source ingestion (PDF + CSV FAQ + TXT) with per-source metadata, incremental re-indexing via content hashing, chat history, a hosted LLM for generation.
2. **From the earlier notebook**: hybrid retrieval (BM25 + semantic + Reciprocal Rank Fusion), cross-encoder reranking, and a code-level grounding check that rejects ungrounded answers before they're returned.

The document collection is dataset-independent: drop any `.pdf`, `.csv` (with `question`/`answer` columns), or `.txt` files into `data/documents/` and rebuild the index.

### Notebook workflow

This notebook builds a complete retrieval-augmented generation (RAG) assistant. It loads local documents, prepares them for search, retrieves relevant context, generates a grounded answer, and runs a small test at the end.

The next cell installs the Python packages used by the notebook: LangChain for orchestration, Chroma for vector storage, embedding and reranking models for retrieval, PDF parsing, and environment-variable support.

In [1]:
%pip install -q langchain langchain-community langchain-huggingface langchain-chroma langchain-groq chromadb rank-bm25 sentence-transformers pypdf pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

This cell imports the required libraries, loads environment variables, identifies the project directories, and defines the models and storage locations used throughout the pipeline. It also creates the document and vector-store directories when needed.

In [4]:
import os
import re
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

load_dotenv()

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "documents"
PERSIST_DIR = BASE_DIR / "data" / "chroma_db"
HASH_FILE = PERSIST_DIR / "source_hash.txt"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "rag_assistant"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
INGEST_VERSION = "v1-hybrid-merged"

# Whether the assistant must cite "Source: <file>" in every answer.
# On for an official FAQ bot (traceability matters); turn off for a
# quick internal assistant where you just want the answer.
INCLUDE_SOURCES_IN_ANSWER = True

print("Documents dir :", DATA_DIR.resolve())
print("Chroma dir    :", PERSIST_DIR.resolve())

C:\Users\PC\AppData\Local\Temp\ipykernel_22324\2299302762.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader
c:\Users\PC\CERIST\intelligent-rag-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documents dir : C:\Users\PC\CERIST\intelligent-rag-assistant\data\documents
Chroma dir    : C:\Users\PC\CERIST\intelligent-rag-assistant\data\chroma_db


## 2. Document loading

Scans `data/documents/` for any PDF, CSV, or TXT file and loads it with source metadata attached.

- **CSV** is treated as a FAQ table if it has `question`/`answer` columns — each row becomes one atomic document (not chunked later, since a Q&A pair shouldn't be split).
- **PDF/TXT** are treated as reference documents and get chunked normally.
- Text cleaning is conservative: it only collapses repeated spaces/tabs and excess blank lines, and never merges separate lines into one. Collapsing all whitespace (including newlines) is what destroys table structure — a table row like `CE  AR  MR  DR` / `02  02  06  01` becomes one ambiguous string that no prompt can reliably unscramble.

The following code defines the document loaders. PDFs are loaded page by page, text files are loaded as reference documents, and FAQ CSV rows are converted into searchable question-and-answer documents. Each document receives metadata such as its source file and type.

In [5]:
def clean_text(text: str) -> str:
    """Conservative whitespace normalization that preserves line structure."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    return text.strip()


def clean_value(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def load_pdf(path: Path) -> list[Document]:
    pages = PyPDFLoader(str(path)).load()
    for i, doc in enumerate(pages):
        doc.page_content = clean_text(doc.page_content)
        doc.metadata.update({
            "doc_type": "reference",
            "source": path.name,
            "source_type": "pdf",
            "page": i + 1,
        })
    return pages


def load_txt(path: Path) -> list[Document]:
    docs = TextLoader(str(path), encoding="utf-8").load()
    for doc in docs:
        doc.page_content = clean_text(doc.page_content)
        doc.metadata.update({
            "doc_type": "reference",
            "source": path.name,
            "source_type": "text",
        })
    return docs


def load_faq_csv(path: Path) -> list[Document]:
    df = pd.read_csv(path)
    if "question" not in df.columns or "answer" not in df.columns:
        raise ValueError(f"{path.name} has no question/answer columns; skipping as FAQ source.")

    docs = []
    for _, row in df.iterrows():
        question = clean_value(row["question"])
        answer = clean_value(row["answer"])
        docs.append(Document(
            page_content=f"Question: {question}\nAnswer: {answer}",
            metadata={
                "doc_type": "faq",
                "source": path.name,
                "source_type": "csv_faq",
                "category": clean_value(row.get("category", "")),
                "disclaimer": clean_value(row.get("disclaimer", "")),
                "atomic": True,   # never chunk this document further
            },
        ))
    return docs


def load_all_documents(data_dir: Path) -> list[Document]:
    documents = []

    for path in sorted(data_dir.glob("*.pdf")):
        documents.extend(load_pdf(path))

    for path in sorted(data_dir.glob("*.txt")):
        documents.extend(load_txt(path))

    for path in sorted(data_dir.glob("*.csv")):
        try:
            documents.extend(load_faq_csv(path))
        except ValueError as e:
            print(f"Skipping {path.name}: {e}")

    return documents


documents = load_all_documents(DATA_DIR)
print(f"Loaded {len(documents)} document(s) from {DATA_DIR}")

Loaded 27 document(s) from c:\Users\PC\CERIST\intelligent-rag-assistant\data\documents


## 3. Chunking

FAQ rows are already atomic (one question/answer pair) and are never split. Only reference documents (PDF/TXT) go through the splitter.

This cell splits long reference documents into overlapping chunks so retrieval can return focused passages. FAQ entries marked as atomic remain intact because each question-and-answer pair should be treated as one unit.

In [6]:
def chunk_documents(docs: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " "],
    )

    atomic_docs = [d for d in docs if d.metadata.get("atomic")]
    splittable_docs = [d for d in docs if not d.metadata.get("atomic")]

    chunked = splitter.split_documents(splittable_docs) if splittable_docs else []

    for i, chunk in enumerate(chunked):
        chunk.metadata["chunk_id"] = f"{chunk.metadata.get('source', 'doc')}_chunk_{i}"

    for i, doc in enumerate(atomic_docs):
        doc.metadata["chunk_id"] = f"{doc.metadata.get('source', 'faq')}_row_{i}"

    return atomic_docs + chunked


chunks = chunk_documents(documents)
print(f"Total chunks: {len(chunks)} ({len(documents)} source documents)")

Total chunks: 71 (27 source documents)


## 4. Vector store (incremental indexing)

Hashes every source file plus the embedding model name and an ingest version string. If nothing changed since the last run, the existing Chroma collection is reused instead of re-embedding everything from scratch.

This cell creates or reuses the Chroma vector store. File hashes detect changes in the source documents, embedding model, or ingestion version, so unchanged data can reuse the existing index instead of being embedded again.

In [7]:
def file_hash(path: Path) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            hasher.update(block)
    return hasher.hexdigest()


def source_signature(data_dir: Path) -> str:
    hasher = hashlib.sha256()
    for path in sorted(data_dir.glob("*")):
        if path.is_file():
            hasher.update(file_hash(path).encode("utf-8"))
    hasher.update(EMBED_MODEL.encode("utf-8"))
    hasher.update(INGEST_VERSION.encode("utf-8"))
    return hasher.hexdigest()


def get_vector_store(chunks: list[Document], embeddings: HuggingFaceEmbeddings) -> Chroma:
    current_sig = source_signature(DATA_DIR)

    if PERSIST_DIR.exists() and HASH_FILE.exists():
        if HASH_FILE.read_text(encoding="utf-8").strip() == current_sig:
            print("No source changes detected — reusing existing index.")
            return Chroma(
                collection_name=COLLECTION_NAME,
                embedding_function=embeddings,
                persist_directory=str(PERSIST_DIR),
            )
        shutil.rmtree(PERSIST_DIR)
        PERSIST_DIR.mkdir(parents=True, exist_ok=True)

    print("Building new index...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(PERSIST_DIR),
    )
    HASH_FILE.write_text(current_sig, encoding="utf-8")
    return vector_store


embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vector_store = get_vector_store(chunks, embeddings)
print("Chunks in vector store:", vector_store._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5181.20it/s]


No source changes detected — reusing existing index.
Chunks in vector store: 71


## 5. Hybrid retrieval: BM25 + semantic + Reciprocal Rank Fusion

Semantic search alone misses exact terms (names, IDs, acronyms). BM25 alone misses paraphrased questions. RRF combines both rankings without needing their scores on the same scale.

This cell builds the hybrid retriever. BM25 finds exact keyword matches, semantic search finds meaning-based matches, Reciprocal Rank Fusion combines both rankings, and a cross-encoder reranks the best candidates for the final context.

In [8]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", text.lower())


bm25_corpus = [tokenize(c.page_content) for c in chunks]
bm25 = BM25Okapi(bm25_corpus)


def retrieve_bm25(query: str, k: int = 8) -> list[dict]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:k]
    return [
        {"chunk_id": chunks[i].metadata["chunk_id"], "doc": chunks[i], "score": float(scores[i])}
        for i in top_idx
    ]


def retrieve_semantic(query: str, k: int = 8) -> list[dict]:
    results = vector_store.similarity_search_with_relevance_scores(query, k=k)
    return [
        {"chunk_id": doc.metadata["chunk_id"], "doc": doc, "score": float(score)}
        for doc, score in results
    ]


RRF_K = 60

def reciprocal_rank_fusion(semantic_results, keyword_results, k=RRF_K) -> list[dict]:
    fused_scores = defaultdict(float)
    data = {}

    for rank, r in enumerate(semantic_results, start=1):
        fused_scores[r["chunk_id"]] += 1 / (k + rank)
        data[r["chunk_id"]] = r["doc"]

    for rank, r in enumerate(keyword_results, start=1):
        fused_scores[r["chunk_id"]] += 1 / (k + rank)
        data.setdefault(r["chunk_id"], r["doc"])

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"chunk_id": cid, "doc": data[cid], "rrf_score": score} for cid, score in ranked]


reranker = CrossEncoder(RERANKER_MODEL)


def retrieve_final(query: str, top_k: int = 5, candidate_k: int = 10) -> list[dict]:
    semantic = retrieve_semantic(query, k=candidate_k)
    keyword = retrieve_bm25(query, k=candidate_k)
    fused = reciprocal_rank_fusion(semantic, keyword)[:candidate_k]

    if not fused:
        return []

    pairs = [[query, r["doc"].page_content] for r in fused]
    scores = reranker.predict(pairs)

    for r, score in zip(fused, scores):
        r["reranker_score"] = float(score)

    fused.sort(key=lambda x: x["reranker_score"], reverse=True)
    return fused[:top_k]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4011.75it/s]


## 6. Grounding check

A code-level safety net: rejects an answer if it shares almost no vocabulary with the retrieved context, catching cases where the model ignored the grounding rule instead of relying on the prompt alone.

This cell adds a lightweight grounding check after generation. It verifies that the answer has meaningful vocabulary overlap with the retrieved context and replaces unsupported answers with the standard knowledge-base response.

In [9]:
def is_answer_supported(answer: str, context: str) -> bool:
    if not answer or not answer.strip():
        return False

    cleaned = answer.strip().lower()
    if cleaned.startswith("i can only answer") or cleaned.startswith("this isn't specified"):
        return True

    answer_tokens = set(re.findall(r"\b[\w'-]+\b", cleaned))
    context_tokens = set(re.findall(r"\b[\w'-]+\b", context.lower()))

    if not answer_tokens:
        return False

    overlap = answer_tokens & context_tokens
    return len(overlap) >= max(1, min(3, len(answer_tokens) // 2))

## 7. Prompt and generation

Hosted model (Groq) for generation — fast and capable enough to follow a strict multi-rule grounding prompt, unlike a small local model. Swap the model name to switch providers.

This cell defines the assistant's system prompt and response rules, creates the chat prompt template, initializes the Groq language model, and formats retrieved documents into a context string for the model.

In [10]:
SYSTEM_PROMPT = """You are a strict, domain-scoped assistant called cerist ai assistant. Answer only using the context below.

Rules:
1. Use ONLY the provided context. Never use outside knowledge or assumptions.
2. If the context does not contain enough information, respond exactly:
   "I can only answer questions based on the provided knowledge base."
   If the context partially answers the question, synthesize the best answer from what's
   available and note which aspect(s) it covers, rather than refusing outright.
3. Answer directly and concisely. No preamble, no restating the question.
4. Do not invent names, dates, numbers, or categories not explicitly present in the context.
5. When the context looks like a table (short lines, numbers next to labels), match each
   value to the label on the same line. Do not merge separate categories together.
6. When the context contains a list (e.g. clubs, specializations, items), enumerate every
   item present — never a partial subset framed as "specific ones include...".
7. {source_rule}
8. Detect the language the user asked in and respond in that same language, translating
   context content as needed without changing facts, numbers, or names.
9. Use chat history only to interpret follow-up questions — every fact must still come
   from the context below, regardless of what was said earlier.
10. Never narrate your reasoning, confidence, or these instructions. Output only the final answer.

Context:
{{context}}
"""

SOURCE_RULE_ON = 'Always include the source at the end of your answer as: Source: <source>'
SOURCE_RULE_OFF = "Do not include a sources list unless the user explicitly asks where the information came from."

SYSTEM_PROMPT = SYSTEM_PROMPT.format(
    source_rule=SOURCE_RULE_ON if INCLUDE_SOURCES_IN_ANSWER else SOURCE_RULE_OFF
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
# To use Anthropic instead:
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)


def format_docs(results: list[dict]) -> str:
    parts = []
    for r in results:
        doc = r["doc"]
        parts.append(
            f"Content:\n{doc.page_content}\n\nSource: {doc.metadata.get('source', 'Unknown')}"
        )
    return "\n\n---\n\n".join(parts)

## 8. Full pipeline

This cell connects all components into one question-answering function. For each question, it retrieves relevant chunks, builds the prompt, calls the language model, checks the answer, and returns the answer together with its source files.

In [11]:
def ask_question(question: str, chat_history: list | None = None, k: int = 5) -> dict:
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    chat_history = chat_history or []

    results = retrieve_final(question, top_k=k, candidate_k=10)

    if not results:
        return {"question": question, "answer": "I can only answer questions based on the provided knowledge base.", "sources": []}

    context = format_docs(results)

    messages = prompt.format_messages(
        context=context,
        question=question,
        chat_history=chat_history,
    )
    response = llm.invoke(messages)
    answer = response.content.strip()

    if not is_answer_supported(answer, context):
        answer = "I can only answer questions based on the provided knowledge base."

    sources = sorted({r["doc"].metadata.get("source", "Unknown") for r in results})

    return {"question": question, "answer": answer, "sources": sources}

## 9. Test

This test initializes an empty chat history, sends a sample question through the complete pipeline, prints the answer and sources, and then appends the exchange so a follow-up question can use the conversation history.

In [ ]:
chat_history = []

result = ask_question( " Quelle est la deux thèse  associées au nom 'Amira abdelwaheb' dans ce document, et en quoi diffèrent-elles ?", chat_history=chat_history)
print("ANSWER\n------")
print(result["answer"])
print("\nSOURCES:", result["sources"])

chat_history.append(HumanMessage(content="What is ESI?"))
chat_history.append(AIMessage(content=result["answer"]))

C:\Users\PC\AppData\Local\Temp\ipykernel_22324\505285034.py:19: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b9dadc8f-d229-43c3-bea3-b1bd4c270084', metadata={'doc_type': 'reference', 'producer': 'Microsoft: Print To PDF', 'moddate': '2025-11-19T14:20:45+01:00', 'creator': 'PyPDF', 'author': '', 'page': 9, 'chunk_id': 'Bilan2025_dsi.pdf_chunk_19', 'source_type': 'pdf', 'source': 'Bilan2025_dsi.pdf', 'page_label': '9', 'creationdate': '2025-11-19T14:20:45+01:00', 'total_pages': 27, 'title': 'Microsoft Word - Bilan2025_dsi'}, page_content='majeure dans notre pays pour exercer cette activité. Toutefois, l’absence de ce mode de paiement\nn’a pas empêché certains pionniers à se lancer dans le e-commerce en Algérie. Même s’il tarde\nà être opérationnel, «il existe de nombreuses solutions alternatives à la carte de crédit, comme\nle paiement par chèque, les cartes pré- payées, ou encore le paiement à la livraison.\nCeci dit, le problème du paiement électronique n’e

ANSWER
------
I can only answer questions based on the provided knowledge base. Source: Bilan2025_dsi.pdf

SOURCES: ['Bilan2025_dsi.pdf']


## 10. Batch question testing

Edit the question list in the next cell to test the assistant with several questions. Each answer, source list, and timestamp is saved in a JSON file under `data/test_results/` so the results can be reviewed or shown later.

In [16]:
from datetime import datetime
import json

questions = [
    # Simple factual retrieval
    'Qui est le chef de projet de la "Plateforme proactive pour la cyber-sécurité basée sur l\'IA et le Bigdata" ?',
    "Quelle est la date de fin prévue du projet e-commerce sécurisé ?",
    "Combien de chercheurs compose le potentiel humain de la division en 2025 ?",

    # Numeric aggregation
    "Combien de publications dans des journaux internationaux la division a-t-elle produites en 2025, et combien sont en cours de soumission ?",
    "Quel est le nombre total de thèses soutenues, en master, et en licence encadrées en 2025 ?",
    "Combien de prototypes et de logiciels ont été produits par la division en 2025 ?",

    # Multi-hop retrieval
    "Quel est le pourcentage d'avancement du projet dont Boulemtafes Amine est le chef, et quelle est sa date de fin prévue ?",
    "Zemmache Amina travaille sur quel projet, et quel article a-t-elle soumis en 2025 ?",
    "Quels projets sont actuellement \"gelés\" ou \"à l'arrêt\", et pour quelle raison ?",

    # Same-name disambiguation
    "Amira Abdelouahab participe à combien de projets différents, et quel est son pourcentage de participation dans chacun ?",
    "Quelles sont les deux thèses différentes associées au nom \"Amira\" dans ce document, et en quoi diffèrent-elles ?",

    # Negative or absence tests
    "Combien de recrutements ont eu lieu dans la division en 2025 ?",
    "Quel est le budget total alloué à la division en 2025 ?",
    "Quels sont les résultats du projet \"Bâtiment Intelligent\" en 2025 ?",

    # Whole-document synthesis
    "Résume les principales difficultés rencontrées par la division en 2025.",
    "Quels sont les projets prévus ou envisagés pour 2027 ?",
    "Quelles sont les partenariats internationaux de la division, et avec quels pays ?",
]

results = []
batch_chat_history = []

for question in questions:
    try:
        result = ask_question(question, chat_history=batch_chat_history)
        results.append({
            "question": question,
            "answer": result["answer"],
            "sources": result["sources"],
            "recorded_at": datetime.now().isoformat(timespec="seconds"),
        })

        batch_chat_history.append(HumanMessage(content=question))
        batch_chat_history.append(AIMessage(content=result["answer"]))

        print(f"\nQUESTION: {question}")
        print(f"ANSWER: {result['answer']}")
        print(f"SOURCES: {', '.join(result['sources']) if result['sources'] else 'None'}")
    except Exception as error:
        results.append({
            "question": question,
            "answer": None,
            "sources": [],
            "error": str(error),
            "recorded_at": datetime.now().isoformat(timespec="seconds"),
        })
        print(f"\nQUESTION: {question}")
        print(f"ERROR: {error}")

results_dir = BASE_DIR / "data" / "test_results"
results_dir.mkdir(parents=True, exist_ok=True)

results_file = results_dir / f"rag_test_results_{datetime.now():%Y%m%d_%H%M%S}.json"
results_file.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"\nSaved {len(results)} test result(s) to: {results_file}")


QUESTION: Qui est le chef de projet de la "Plateforme proactive pour la cyber-sécurité basée sur l'IA et le Bigdata" ?
ANSWER: Guemraoui Lila  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Quelle est la date de fin prévue du projet e-commerce sécurisé ?
ANSWER: Décembre 2025  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Combien de chercheurs compose le potentiel humain de la division en 2025 ?
ANSWER: 15 chercheurs  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Combien de publications dans des journaux internationaux la division a-t-elle produites en 2025, et combien sont en cours de soumission ?
ANSWER: 4 publications dans des journaux internationaux ; 7 sont en cours de soumission.  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Quel est le nombre total de thèses soutenues, en master, et en licence encadrées en 2025 ?
ANSWER: Doctorat : 9 ; Master : 10 ; Licence : 2  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan

C:\Users\PC\AppData\Local\Temp\ipykernel_22324\505285034.py:19: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='30d0766e-60bc-45a8-8ed1-b447791fedb7', metadata={'title': 'Microsoft Word - Bilan2025_dsi', 'source_type': 'pdf', 'author': '', 'total_pages': 27, 'source': 'Bilan2025_dsi.pdf', 'doc_type': 'reference', 'chunk_id': 'Bilan2025_dsi.pdf_chunk_68', 'creationdate': '2025-11-19T14:20:45+01:00', 'moddate': '2025-11-19T14:20:45+01:00', 'page_label': '26', 'creator': 'PyPDF', 'producer': 'Microsoft: Print To PDF', 'page': 26}, page_content='5. RESULTATS QUANTIFIES :\nRésultats obtenus en 2025 : gelé pour indisponibilité de l’équipe.'), 0.15834876154279276), (Document(id='1f48b812-04d6-4484-8f56-617b42b5563b', metadata={'source_type': 'pdf', 'creationdate': '2025-11-19T14:20:45+01:00', 'title': 'Microsoft Word - Bilan2025_dsi', 'total_pages': 27, 'moddate': '2025-11-19T14:20:45+01:00', 'creator': 'PyPDF', 'chunk_id': 'Bilan2025_dsi.pdf_chunk_21', 'doc_type': '


QUESTION: Quels projets sont actuellement "gelés" ou "à l'arrêt", et pour quelle raison ?
ANSWER: - **Automatisation de l’analyse de la Sécurité d’un Bâtiment Intelligent (Smart Building Security Analysis Automation)** – gelé depuis 2022 faute de personnel technique disponible.  
- **Infrastructure Big Data** – à l’arrêt car l’équipe responsable (d’une autre division) n’est pas disponible.  
- **Dispositifs de Sécurité** (travaux de l’équipe des Dispositifs de Sécurité) – interrompus en raison d’un manque de ressources (ingénieurs, adresses IP publiques, etc.).  

Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Amira Abdelouahab participe à combien de projets différents, et quel est son pourcentage de participation dans chacun ?
ANSWER: Amira Abdelouahab participe à **2 projets différents** :

1. **Architecture sécurisée pour une application d’apprentissage fédéré dans le domaine de la santé** – le bilan indique deux valeurs : 10 % de participation (premier état) et 50

C:\Users\PC\AppData\Local\Temp\ipykernel_22324\505285034.py:19: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b9dadc8f-d229-43c3-bea3-b1bd4c270084', metadata={'title': 'Microsoft Word - Bilan2025_dsi', 'moddate': '2025-11-19T14:20:45+01:00', 'page': 9, 'chunk_id': 'Bilan2025_dsi.pdf_chunk_19', 'producer': 'Microsoft: Print To PDF', 'page_label': '9', 'creator': 'PyPDF', 'source_type': 'pdf', 'creationdate': '2025-11-19T14:20:45+01:00', 'doc_type': 'reference', 'source': 'Bilan2025_dsi.pdf', 'total_pages': 27, 'author': ''}, page_content='majeure dans notre pays pour exercer cette activité. Toutefois, l’absence de ce mode de paiement\nn’a pas empêché certains pionniers à se lancer dans le e-commerce en Algérie. Même s’il tarde\nà être opérationnel, «il existe de nombreuses solutions alternatives à la carte de crédit, comme\nle paiement par chèque, les cartes pré- payées, ou encore le paiement à la livraison.\nCeci dit, le problème du paiement électronique n’e


QUESTION: Quelles sont les deux thèses différentes associées au nom "Amira" dans ce document, et en quoi diffèrent-elles ?
ANSWER: I can only answer questions based on the provided knowledge base.
SOURCES: Bilan2025_dsi.pdf

QUESTION: Combien de recrutements ont eu lieu dans la division en 2025 ?
ANSWER: 0 recrutement  
Source: Bilan2025_dsi.pdf
SOURCES: Bilan2025_dsi.pdf

QUESTION: Quel est le budget total alloué à la division en 2025 ?
ANSWER: I can only answer questions based on the provided knowledge base.
SOURCES: Bilan2025_dsi.pdf

QUESTION: Quels sont les résultats du projet "Bâtiment Intelligent" en 2025 ?
ANSWER: I can only answer questions based on the provided knowledge base.
SOURCES: Bilan2025_dsi.pdf

QUESTION: Résume les principales difficultés rencontrées par la division en 2025.
ANSWER: Les principales difficultés rencontrées par la division en 2025 se résument à un manque de personnel technique, ce qui a limité le recrutement, entraîné des projets gelés ou à l’arrêt e